Data Ingestion

In [6]:
### Document structure
import os
from langchain_core.documents import Document

In [5]:
doc=Document(
    page_content="This is the main text content I am using to create RAG",
    metadata={
        "source":"example.txt",
        "pages":1,
        "author":"Lingaraj",
        "date_created":"23-05-2026"
    }
)
doc

Document(metadata={'source': 'example.txt', 'pages': 1, 'author': 'Lingaraj', 'date_created': '23-05-2026'}, page_content='This is the main text content I am using to create RAG')

In [21]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

### Read all the pdf's inside the directory
def process_all_pdfs_fast(pdf_directory):
    # DirectoryLoader handles the recursive search and loading automatically
    loader = DirectoryLoader(
        pdf_directory, 
        glob="./**/*.pdf", 
        loader_cls=PyMuPDFLoader,
        show_progress=True
    )
    
    # Load all documents into memory
    documents = loader.load()
    
    # Optional: If you need custom metadata updates like 'file_type'
    for doc in documents:
        doc.metadata['file_type'] = 'pdf'
        # PyMuPDFLoader already adds 'source' to metadata automatically
        
    print(f"\nTotal documents loaded: {len(documents)}")
    return documents

# Usage
all_pdf_documents = process_all_pdfs_fast("../data")

100%|██████████| 4/4 [00:00<00:00, 35.24it/s]


Total documents loaded: 20


In [18]:
all_pdf_documents

[Document(metadata={'producer': '', 'creator': '', 'creationdate': '', 'source': '..\\data\\pdf\\2659003.pdf', 'file_path': '..\\data\\pdf\\2659003.pdf', 'total_pages': 15, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 0, 'file_type': 'pdf'}, page_content='LAS VEGAS  METROPOLIT\n AN POLICE  DEPARTMENT\nIMPAIRED\n DRMNG\n REPORT\nEvent  Number:\n.[). Ncimber\nLLV240200021172\nJ\'.51l\'j}\nTHE UNDERSIGNED  MAKES  THE FOLLOWING  DECLARATIONS\n SUBJECT  TO THE PENALTY  FOR PERJURY  AND SAYS:\nThat  I am  a Police  Officer\n with  the  Las  Vegas\n Metropolitan\n Police  Department,\n Clark  County,\n Nevada\n being  so employed\nfor  a period  of 3 years  \nThat  I learned\n the  following\n facts  and  circumstances\n which  led me  to believe\n that  the below\nsciblect committed  (or was commithng)  tlie offense  of @ Felony Driving Under  The Inflbience (DUI) [gl Misdemeanot\nDrivi

In [22]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [23]:
chunks=split_documents(all_pdf_documents)
chunks

Split 20 documents into 54 chunks

Example chunk:
Content: LAS VEGAS  METROPOLIT
 AN POLICE  DEPARTMENT
IMPAIRED
 DRMNG
 REPORT
Event  Number:
.[). Ncimber
LLV240200021172
J'.51l'j}
THE UNDERSIGNED  MAKES  THE FOLLOWING  DECLARATIONS
 SUBJECT  TO THE PENALTY ...
Metadata: {'producer': '', 'creator': '', 'creationdate': '', 'source': '..\\data\\pdf\\2659003.pdf', 'file_path': '..\\data\\pdf\\2659003.pdf', 'total_pages': 15, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 0, 'file_type': 'pdf'}


[Document(metadata={'producer': '', 'creator': '', 'creationdate': '', 'source': '..\\data\\pdf\\2659003.pdf', 'file_path': '..\\data\\pdf\\2659003.pdf', 'total_pages': 15, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 0, 'file_type': 'pdf'}, page_content="LAS VEGAS  METROPOLIT\n AN POLICE  DEPARTMENT\nIMPAIRED\n DRMNG\n REPORT\nEvent  Number:\n.[). Ncimber\nLLV240200021172\nJ'.51l'j}\nTHE UNDERSIGNED  MAKES  THE FOLLOWING  DECLARATIONS\n SUBJECT  TO THE PENALTY  FOR PERJURY  AND SAYS:\nThat  I am  a Police  Officer\n with  the  Las  Vegas\n Metropolitan\n Police  Department,\n Clark  County,\n Nevada\n being  so employed\nfor  a period  of 3 years  \nThat  I learned\n the  following\n facts  and  circumstances\n which  led me  to believe\n that  the below\nsciblect committed  (or was commithng)  tlie offense  of @ Felony Driving Under  The Inflbience (DUI) [gl Misdemeanot\nDriving

Embedding

In [1]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

c:\Users\HP\Documents\MYRAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings
    
## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


c:\Users\HP\Documents\MYRAG\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\HP\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4425.32it/s]


Model loaded successfully. Embedding dimension: 384


Vector Store

In [7]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [24]:
chunks

[Document(metadata={'producer': '', 'creator': '', 'creationdate': '', 'source': '..\\data\\pdf\\2659003.pdf', 'file_path': '..\\data\\pdf\\2659003.pdf', 'total_pages': 15, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 0, 'file_type': 'pdf'}, page_content="LAS VEGAS  METROPOLIT\n AN POLICE  DEPARTMENT\nIMPAIRED\n DRMNG\n REPORT\nEvent  Number:\n.[). Ncimber\nLLV240200021172\nJ'.51l'j}\nTHE UNDERSIGNED  MAKES  THE FOLLOWING  DECLARATIONS\n SUBJECT  TO THE PENALTY  FOR PERJURY  AND SAYS:\nThat  I am  a Police  Officer\n with  the  Las  Vegas\n Metropolitan\n Police  Department,\n Clark  County,\n Nevada\n being  so employed\nfor  a period  of 3 years  \nThat  I learned\n the  following\n facts  and  circumstances\n which  led me  to believe\n that  the below\nsciblect committed  (or was commithng)  tlie offense  of @ Felony Driving Under  The Inflbience (DUI) [gl Misdemeanot\nDriving

In [25]:
## Text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 54 texts...


Batches: 100%|██████████| 2/2 [00:01<00:00,  1.22it/s]

Generated embeddings with shape: (54, 384)
Adding 54 documents to vector store...
Successfully added 54 documents to vector store
Total documents in collection: 54


Retreival pipeline

In [26]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)



In [27]:
rag_retriever

In [39]:
rag_retriever.retrieve("what was the status of the fine?")

Retrieving documents for query: 'what was the status of the fine?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 80.04it/s]

Generated embeddings with shape: (1, 384)
Retrieved 1 documents (after filtering)


[{'id': 'doc_e9234bb7_53',
  'content': 'Motion - Miscellaneous Filed\nFines, Fees & Sentencing\nItem Name\nDue\nStatus/Paid\nBalance\nCompleted Date\nSuspended Jail (2 Days CTS)\n179 - Days Suspended\nImposed\n0\n7/25/2024\nStay Out of Trouble - Broad\n1 - Year(s)\nImposed\n0\n7/25/2024\nDUI Program\nImposed\n0\n7/25/2024\nFine\n$910.00\nImposed\n0\n7/25/2024\nVictim Impact Panel Class\nImposed\n0\n7/25/2024\nNo Traffic Offenses\nImposed\n0\n7/25/2024\nGeneral Assessment Fines\n$910.00\n$.00\n$910.00\n7/25/2024\n7/30/24, 12:07 PM\nCity of Las Vegas > Government > Municipal Court > Case Docket Search\nhttps://www.lasvegasnevada.gov/Government/Municipal-Court/Case-Docket-Search#/about/24-005706\n2/2',
  'metadata': {'keywords': '',
   'creationDate': "D:20240730190752+00'00'",
   'source': '..\\data\\pdf\\City of Las Vegas _ Government _ Municipal Court _ Case Docket Search (1).pdf',
   'format': 'PDF 1.4',
   'producer': 'Skia/PDF m125',
   'moddate': '2024-07-30T19:07:52+00:00',
   't

VectorDB to LLM

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

print(os.getenv("OPENAI_API_KEY"))

In [49]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage

In [54]:
class OpenAILLM:
    def __init__(self, model_name: str = "gpt-4o", api_key: str =None):
        """
        Initialize OpenAI LLM
        
        Args:
            model_name: OpenAI model name
            api_key: OpenAI API key
        """
        self.model_name = model_name
        self.api_key = api_key or os.environ.get("OPENAI_API_KEY")
        
        if not self.api_key:
            raise ValueError("OPENAI API KEY is required. Set OPENAI_API_KEY environment variable or pass api_key parameter.")
        
        self.llm = ChatOpenAI(
            api_key=self.api_key,
            model_name=self.model_name,
            temperature=0.1,
            max_tokens=1024
        )
        
        print(f"Initialized OpenAI LLM with model: {self.model_name}")

    def generate_response(self, query: str, context: str, max_length: int = 500) -> str:
        """
        Generate response using retrieved context
        
        Args:
            query: User question
            context: Retrieved document context
            max_length: Maximum response length
            
        Returns:
            Generated response string
        """
        
        # Create prompt template
        prompt_template = PromptTemplate(
            input_variables=["context", "question"],
            template="""You are a helpful AI assistant. Use the following context to answer the question accurately and concisely.

Context:
{context}

Question: {question}

Answer: Provide a clear and informative answer based on the context above. If the context doesn't contain enough information to answer the question, say so."""
        )
        
        # Format the prompt
        formatted_prompt = prompt_template.format(context=context, question=query)
        
        try:
            # Generate response
            messages = [HumanMessage(content=formatted_prompt)]
            response = self.llm.invoke(messages)
            return response.content
            
        except Exception as e:
            return f"Error generating response: {str(e)}"
        
    def generate_response_simple(self, query: str, context: str) -> str:
        """
        Simple response generation without complex prompting
        
        Args:
            query: User question
            context: Retrieved context
            
        Returns:
            Generated response
        """
        simple_prompt = f"""Based on this context: {context}

Question: {query}

Answer:"""
        
        try:
            messages = [HumanMessage(content=simple_prompt)]
            response = self.llm.invoke(messages)
            return response.content
        except Exception as e:
            return f"Error: {str(e)}"
    


In [55]:
# Initialize ApenAI LLM
try:
    openai_llm = OpenAILLM(api_key=os.getenv("OPENAI_API_KEY"))
    print("OpenAI LLM initialized successfully!")
except ValueError as e:
    print(f"Warning: {e}")
    print("Please set your OPENAI_API_KEY environment variable to use the LLM.")
    openai_llm = None

Initialized OpenAI LLM with model: gpt-4o
OpenAI LLM initialized successfully!


In [56]:
rag_retriever.retrieve("what was the status of the fine?")

Retrieving documents for query: 'what was the status of the fine?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 84.50it/s]

Generated embeddings with shape: (1, 384)
Retrieved 1 documents (after filtering)


[{'id': 'doc_e9234bb7_53',
  'content': 'Motion - Miscellaneous Filed\nFines, Fees & Sentencing\nItem Name\nDue\nStatus/Paid\nBalance\nCompleted Date\nSuspended Jail (2 Days CTS)\n179 - Days Suspended\nImposed\n0\n7/25/2024\nStay Out of Trouble - Broad\n1 - Year(s)\nImposed\n0\n7/25/2024\nDUI Program\nImposed\n0\n7/25/2024\nFine\n$910.00\nImposed\n0\n7/25/2024\nVictim Impact Panel Class\nImposed\n0\n7/25/2024\nNo Traffic Offenses\nImposed\n0\n7/25/2024\nGeneral Assessment Fines\n$910.00\n$.00\n$910.00\n7/25/2024\n7/30/24, 12:07 PM\nCity of Las Vegas > Government > Municipal Court > Case Docket Search\nhttps://www.lasvegasnevada.gov/Government/Municipal-Court/Case-Docket-Search#/about/24-005706\n2/2',
  'metadata': {'doc_index': 53,
   'source': '..\\data\\pdf\\City of Las Vegas _ Government _ Municipal Court _ Case Docket Search (1).pdf',
   'author': '',
   'format': 'PDF 1.4',
   'page': 1,
   'creationDate': "D:20240730190752+00'00'",
   'title': 'City of Las Vegas > Government > Mu

Integration VectorDB context to LLM Output

In [58]:
### Simple RAG pipeline with Groq LLM
from langchain_openai import ChatOpenAI
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the OpenAI LLM (set your OPENAI_API_KEY in environment)
openai_api_key = os.getenv("OPENAI_API_KEY")

llm=ChatOpenAI(api_key=openai_api_key,model_name="gpt-4o",temperature=0.1,max_tokens=1024)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answwer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content

In [ ]:
answer=rag_simple("what was the status of the fine?",rag_retriever,llm)
print(answer)